# TD3 on LunarLanderContinuous

The TD3 part of `td3.py` (Twin Delayed DDPG, Fujimoto et al., 2018) with the two options of our study:

- **LayerNorm in the critics** (`critic_ln = 1`);
- **the weighted target** `beta * min(Q1', Q2') + (1 - beta) * max(Q1', Q2')` (`beta = 1` is standard TD3);

plus the Monte Carlo measurement of the critic's overestimation bias.

The code, the defaults and the order of random draws are the same as in `td3.py`: on the PPTI machine used
for the report, seed 0 reproduces the report's TD3 run exactly. On another computer the first evaluation still
matches, but tiny floating-point differences between CPUs then make the run diverge: it is an equally valid
run, not an identical one.

In [ ]:
import json, os, time
from types import SimpleNamespace

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Settings

All hyper-parameters of Table 1 in the report. `QUICK = True` trains for 30k steps (a few minutes)
instead of 300k (30–60 minutes on one CPU core).

In [ ]:
QUICK = False

cfg = SimpleNamespace(
    algo="td3", env="LunarLanderContinuous-v3", seed=0,
    total_steps=30_000 if QUICK else 300_000,
    learning_starts=10_000,             # warm-up: random actions before learning starts
    buffer_size=1_000_000, batch_size=256,
    gamma=0.99,                         # discount
    tau=0.005,                          # target networks: theta' <- (1 - tau) theta' + tau theta
    actor_lr=3e-4, critic_lr=3e-4,
    hidden=256, n_layers=2,
    critic_ln=0,                        # 1 = LayerNorm in the critics' hidden layers
    actor_ln=0,
    utd=1,                              # gradient steps per environment step
    policy_delay=2,                     # TD3 change 3: critic updates per actor update
    expl_noise=0.1,                     # Gaussian exploration noise on the actions
    target_noise=0.2, noise_clip=0.5,   # TD3 change 2: target policy smoothing
    beta=1.0,                           # TD3 change 1: beta*min + (1-beta)*max of the two critics
    eval_every=10_000, eval_episodes=10,
    mc_min_horizon=500,                 # bias: skip the last 500 states of time-limit episodes
)

## 2. Networks

- **Actor** `mu(s)`: 8 → 256 → 256 → 2, with `tanh` so that actions are in [−1, 1].
- **Critic**: two independent Q-networks `Q1`, `Q2`, each (s, a) → 256 → 256 → 1.
  With `critic_ln = 1`, every hidden layer is `Linear → LayerNorm → ReLU`.

In [ ]:
def mlp(inp, out, hidden, n_layers, ln):
    layers, d = [], inp
    for _ in range(n_layers):
        layers.append(nn.Linear(d, hidden))
        if ln:
            layers.append(nn.LayerNorm(hidden))
        layers.append(nn.ReLU())
        d = hidden
    layers.append(nn.Linear(d, out))
    return nn.Sequential(*layers)


class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, cfg):
        super().__init__()
        self.net = mlp(obs_dim, act_dim, cfg.hidden, cfg.n_layers, cfg.actor_ln)

    def forward(self, s):
        return torch.tanh(self.net(s))  # LunarLander actions are in [-1, 1]


class Critic(nn.Module):
    # The two Q-networks of TD3, evaluated together: output shape (batch, 2).
    def __init__(self, obs_dim, act_dim, cfg):
        super().__init__()
        self.qs = nn.ModuleList(
            mlp(obs_dim + act_dim, 1, cfg.hidden, cfg.n_layers, cfg.critic_ln) for _ in range(2))

    def forward(self, s, act):
        x = torch.cat([s, act], -1)
        return torch.cat([q(x) for q in self.qs], -1)

## 3. Replay buffer

Stores every transition `(s, a, r, s', d)`; each update samples a random minibatch.
`d = 1` only for a real end of the task (crash or landing), not for the 1000-step time limit.

In [ ]:
class ReplayBuffer:
    def __init__(self, obs_dim, act_dim, size):
        self.s = np.zeros((size, obs_dim), np.float32)
        self.a = np.zeros((size, act_dim), np.float32)
        self.r = np.zeros((size, 1), np.float32)
        self.s2 = np.zeros((size, obs_dim), np.float32)
        self.d = np.zeros((size, 1), np.float32)
        self.size, self.ptr, self.n = size, 0, 0

    def add(self, s, a, r, s2, d):
        i = self.ptr
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, d
        self.ptr = (i + 1) % self.size
        self.n = min(self.n + 1, self.size)

    def sample(self, rng, bs):
        idx = rng.integers(0, self.n, bs)
        return [torch.as_tensor(x[idx]) for x in (self.s, self.a, self.r, self.s2, self.d)]

## 4. Evaluation and overestimation bias

Episodes with the **deterministic** policy (no exploration noise):

- the undiscounted episode return measures **performance**;
- for every visited `(s_t, a_t)`, the discounted Monte Carlo return `G_t` is a sample of the true `Q(s_t, a_t)`,
  so **bias** = mean of `Q1(s_t, a_t) − G_t` (positive = overestimation).

In [ ]:
@torch.no_grad()
def evaluate(env, actor, critic, cfg, rng):
    returns, qs, gs = [], [], []
    for _ in range(cfg.eval_episodes):
        s, _ = env.reset(seed=int(rng.integers(2**31)))
        S, A, R = [], [], []
        done = trunc = False
        while not (done or trunc):
            act = actor(torch.as_tensor(s, dtype=torch.float32)[None])[0].numpy()
            S.append(s); A.append(act)
            s, r, done, trunc, _ = env.step(act)
            R.append(r)
        returns.append(float(np.sum(R)))
        G = np.zeros(len(R))
        g = 0.0
        for t in reversed(range(len(R))):
            g = R[t] + cfg.gamma * g
            G[t] = g
        # A time-limit episode has no exact MC return near its end: drop those states.
        keep = len(R) if done else max(0, len(R) - cfg.mc_min_horizon)
        if keep:
            q = critic(torch.as_tensor(np.array(S[:keep]), dtype=torch.float32),
                       torch.as_tensor(np.array(A[:keep]), dtype=torch.float32))
            qs.append(q.numpy()); gs.append(G[:keep])
    out = {"return_mean": float(np.mean(returns)), "returns": returns}
    if gs:
        Q = np.concatenate(qs); G = np.concatenate(gs)
        out.update({
            "q_mean": float(Q[:, 0].mean()), "mc_mean": float(G.mean()),
            "bias": float((Q[:, 0] - G).mean()),               # Q1: the critic the actor follows
            "bias_min": float((Q.min(1) - G).mean()),
            "q_gap": float((Q.max(1) - Q.min(1)).mean()),       # disagreement of the two critics
        })
    return out

## 5. Setup

Seeds, environments, networks and their **target networks** (slow copies used to compute the TD target).
Keep this order: it fixes the sequence of random draws, which is what makes runs reproducible.

In [ ]:
torch.set_num_threads(1)            # one thread: faster for small networks, and reproducible
np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
rng = np.random.default_rng(cfg.seed)                 # exploration noise, replay sampling
eval_rng = np.random.default_rng(cfg.seed + 10_000)   # evaluation episodes

env = gym.make(cfg.env)
eval_env = gym.make(cfg.env)
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
env.action_space.seed(cfg.seed)

actor = Actor(obs_dim, act_dim, cfg)
critic = Critic(obs_dim, act_dim, cfg)
actor_t = Actor(obs_dim, act_dim, cfg); actor_t.load_state_dict(actor.state_dict())
critic_t = Critic(obs_dim, act_dim, cfg); critic_t.load_state_dict(critic.state_dict())
for p in (*actor_t.parameters(), *critic_t.parameters()):
    p.requires_grad_(False)
actor_opt = torch.optim.Adam(actor.parameters(), lr=cfg.actor_lr)
critic_opt = torch.optim.Adam(critic.parameters(), lr=cfg.critic_lr)
buf = ReplayBuffer(obs_dim, act_dim, min(cfg.buffer_size, cfg.total_steps))

## 6. One TD3 update

TD3 = DDPG + three changes, marked (1)–(3) below:

1. **Clipped double Q**: the target uses the smaller of the two target critics
   (with `beta < 1`, a mix of the smaller and the larger one).
2. **Target policy smoothing**: clipped noise on the target action `mu'(s')`.
3. **Delayed updates**: the actor and the target networks change only every `policy_delay` critic updates.

Both critics regress on the same TD target `y = r + gamma (1 − d) Q'(s', a')`; the actor maximises `Q1`.

In [ ]:
n_critic_updates = 0


def td3_update():
    global n_critic_updates
    bs, ba, br, bs2, bd = buf.sample(rng, cfg.batch_size)
    with torch.no_grad():
        a2 = actor_t(bs2)
        noise = (torch.randn_like(a2) * cfg.target_noise).clamp(-cfg.noise_clip, cfg.noise_clip)
        a2 = (a2 + noise).clamp(-1, 1)                                   # (2) target policy smoothing
        q2 = critic_t(bs2, a2)                                            # Q1', Q2' side by side
        q2 = (cfg.beta * q2.min(1, keepdim=True).values                  # (1) clipped double Q
              + (1 - cfg.beta) * q2.max(1, keepdim=True).values)
        y = br + cfg.gamma * (1 - bd) * q2                                # TD target
    q = critic(bs, ba)
    critic_loss = F.mse_loss(q, y.expand_as(q)) * q.shape[1]             # sum over the two critics
    critic_opt.zero_grad(); critic_loss.backward(); critic_opt.step()
    n_critic_updates += 1

    actor_loss = None
    if n_critic_updates % cfg.policy_delay == 0:                          # (3) delayed updates
        actor_loss = -critic(bs, actor(bs))[:, 0].mean()                  # climb Q1
        actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()
        with torch.no_grad():                                             # soft target update
            for net, tgt in ((actor, actor_t), (critic, critic_t)):
                for p, pt in zip(net.parameters(), tgt.parameters()):
                    pt.lerp_(p, cfg.tau)
        actor_loss = actor_loss.item()
    return critic_loss.item(), actor_loss, q[:, 0].mean().item()

## 7. Training loop

Each environment step: act (random during the warm-up, then actor + Gaussian noise), store the transition,
then `utd` TD3 updates. Every `eval_every` steps: evaluation and bias measurement.

In [ ]:
log = {"config": vars(cfg), "evals": [], "train_episodes": []}
critic_loss = actor_loss = q_batch = float("nan")
t0 = time.time()
s, _ = env.reset(seed=cfg.seed)
ep_ret, ep_len = 0.0, 0

for step in range(1, cfg.total_steps + 1):
    if step <= cfg.learning_starts:
        act = env.action_space.sample()
    else:
        with torch.no_grad():
            act = actor(torch.as_tensor(s, dtype=torch.float32)[None])[0].numpy()
        act = np.clip(act + rng.normal(0, cfg.expl_noise, act_dim), -1, 1).astype(np.float32)
    s2, r, done, trunc, _ = env.step(act)
    buf.add(s, act, r, s2, float(done))       # bootstrap through time limits, not through real ends
    s = s2; ep_ret += r; ep_len += 1
    if done or trunc:
        log["train_episodes"].append((step, ep_ret, ep_len))
        s, _ = env.reset(); ep_ret, ep_len = 0.0, 0

    if step > cfg.learning_starts:
        for _ in range(cfg.utd):
            critic_loss, new_actor_loss, q_batch = td3_update()
            if new_actor_loss is not None:
                actor_loss = new_actor_loss

    if step % cfg.eval_every == 0:
        ev = evaluate(eval_env, actor, critic, cfg, eval_rng)
        ev.update(step=step, critic_loss=critic_loss, actor_loss=actor_loss, q_batch=q_batch,
                  time=time.time() - t0)
        log["evals"].append(ev)
        print(f"step {step:6d}   return {ev['return_mean']:7.1f}   bias {ev.get('bias', float('nan')):7.2f}   "
              f"Q {ev.get('q_mean', float('nan')):7.1f}   MC {ev.get('mc_mean', float('nan')):7.1f}   "
              f"{ev['time']:5.0f}s", flush=True)

## 8. Results of this run

Left: evaluation return (dotted line: 200 = solved). Right: the critic's average estimate `Q1(s, a)` and the
true Monte Carlo return `G` on the same states; the gap between them is the bias.
The evaluation at the end of the warm-up is left out of the right panel (the critic has not been trained yet).

In [ ]:
steps = [e["step"] / 1e3 for e in log["evals"]]
trained = [e for e in log["evals"] if e["step"] > cfg.learning_starts and "bias" in e]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(steps, [e["return_mean"] for e in log["evals"]], color="#2a78d6", lw=2)
ax[0].axhline(200, color="#9a9a96", lw=0.8, ls=":")
ax[0].set(xlabel="Environment steps (×1000)", ylabel="Evaluation return", title="Performance")
ax[1].plot([e["step"] / 1e3 for e in trained], [e["q_mean"] for e in trained], color="#eb6834", lw=2,
           label="critic Q1(s, a)")
ax[1].plot([e["step"] / 1e3 for e in trained], [e["mc_mean"] for e in trained], color="#2a78d6", lw=2,
           label="Monte Carlo return G")
ax[1].set(xlabel="Environment steps (×1000)", ylabel="Value", title="Critic estimate vs true return")
ax[1].legend(frameon=False)
for a_ in ax:
    a_.spines[["top", "right"]].set_visible(False)
    a_.grid(axis="y", color="#e4e4e0")
plt.tight_layout()
plt.show()

## 9. Save the run (optional)

Same JSON format as `td3.py`, so `python analyze.py runs/notebook` can read it.

In [ ]:
name = (f"td3_ln{cfg.critic_ln}{cfg.actor_ln}_L{cfg.n_layers}_utd{cfg.utd}_pd{cfg.policy_delay}"
        + (f"_b{cfg.beta:g}" if cfg.beta != 1 else "") + f"_s{cfg.seed}")
os.makedirs("runs/notebook", exist_ok=True)
with open(f"runs/notebook/{name}.json", "w") as f:
    json.dump(log, f)
print("saved", f"runs/notebook/{name}.json")